In [4]:
# This program is a guided demonstration on using Monte Carlo simulations
# with quantum circuits. Throughout the program there are comments breaking down
# each step and explaining how the classical steps are translated to a quantum circuit.

# The goal of this program is to predict the future amount of money
# using the formula for compounding interest. The program does this using
# classical Monte Carlo simulations, then does the same prediction using Quantum circuit.

In [5]:
# Install the qiskit-finance library, which includes tools
# for modeling financial problems, solving optimization problems, and
# analyzing financial risk.
!pip install qiskit-finance

  Using cached scipy-1.15.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Preparing metadata (setup.py) ... done
  Using cached rustworkx-0.16.0-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (10 kB)
  Using cached sympy-1.13.3-py3-none-any.whl.metadata (12 kB)
  Using cached dill-0.3.9-py3-none-any.whl.metadata (10 kB)
  Using cached stevedore-5.4.1-py3-none-any.whl.metadata (2.3 kB)
  Using cached symengine-0.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.2 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 6.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 34.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pypr

In [ ]:
# Install Qiskit Aer, the high-performance simulator framework for quantum circuits.
# Aer allows us to simulate quantum behavior (like measurement and interference)
# without needing access to real quantum hardware.
# This is especially useful when testing algorithms such as Amplitude Estimation
# on local machines using the 'qasm_simulator' backend.
!pip install qiskit_aer

  Using cached qiskit_aer-0.17.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (8.2 kB)
  Using cached qiskit-2.0.0-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached scipy-1.15.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached rustworkx-0.16.0-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (10 kB)
  Using cached sympy-1.13.3-py3-none-any.whl.metadata (12 kB)
  Using cached dill-0.3.9-py3-none-any.whl.metadata (10 kB)
  Using cached stevedore-5.4.1-py3-none-any.whl.metadata (2.3 kB)
  Using cached symengine-0.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.2 kB)
  Using cached pbr-6.1.1-py2.py3-none-any.whl.metadata (3.4 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached qiskit_aer-0.17.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.4 MB)
Using cached qiskit-2.0.0-cp39-abi3-manylinux_2_17_x86

In [6]:
# ------------------------------------------------------------
# Classical parameters
# ------------------------------------------------------------
# These parameters define the investment scenario and the 
# number of samples for Monte Carlo simulations.
principal = 100000         # Initial investment (present value)
rate = 0.095               # Expected annual growth rate (9.5%)
T = 30                     # Time horizon in years
N = 10**6                  # Number of samples for classical Monte Carlo

# ------------------------------------------------------------
# 1) Direct Classical Calculation
# ------------------------------------------------------------
# Uses the standard compound interest formula to get a precise answer.
final_amount = principal * (1 + rate)**T
print("Classical final amount (direct formula): ${:,.0f}".format(final_amount))
# E.g., ~ $1,555,031 for the given parameters

# ------------------------------------------------------------
# 2) Classical Monte Carlo Approach
# ------------------------------------------------------------
# We'll model the problem as integrating the function f(x) = offset + slope * x,
# where offset = principal and slope = principal * ((1 + rate)**T - 1).
# The uniform random x simulates different 'scenarios' in [0,1].

offset = principal
slope = principal * ((1 + rate)**T - 1)
exact_value = offset + slope * 0.5  # Theoretically expected average if x is uniform in [0,1]

# Classical Monte Carlo simulation
start_time = time.time()
samples = np.random.rand(N)  # Generate N random values in [0, 1]
classical_mc = offset + slope * np.mean(samples)  # Average outcome
end_time = time.time()

print("Classical Monte Carlo estimate: ${:,.0f}".format(classical_mc))
print("Classical MC time: {:.4f} seconds, samples used: {}".format(end_time - start_time, N))

# ------------------------------------------------------------
# 3) Quantum Amplitude Estimation
# ------------------------------------------------------------
# Next, we use Qiskit's Amplitude Estimation to demonstrate a 
# quantum-based approach to the same problem.

# Define a post-processing function that translates the amplitude into the final value
def post_processing(a):
    return offset + slope * a

# Create a simple 1-qubit circuit that always outputs |1>
# This means amplitude a=1, and post_processing(1) would be offset + slope*1.
qc = QuantumCircuit(1)
qc.x(0)  # Flip qubit to |1>

# Declare which qubits represent our objective measurement
objective_qubits = [0]

# Create the EstimationProblem object that configures amplitude estimation
problem = EstimationProblem(
    state_preparation=qc,       # The quantum circuit that prepares the state
    objective_qubits=objective_qubits,
    post_processing=post_processing
)

# Number of evaluation qubits (2^num_eval_qubits = # of evaluations)
# More qubits -> more precision but more overhead
num_eval_qubits = 3

# Use Qiskit's built-in quantum simulator backend
quantum_instance = Aer.get_backend('qasm_simulator')

# Set up the amplitude estimation algorithm
ae = AmplitudeEstimation(num_eval_qubits=num_eval_qubits)
ae.quantum_instance = quantum_instance

# Measure runtime for quantum approach
start_time_q = time.time()

# Execute the quantum amplitude estimation
result = ae.estimate(problem)
end_time_q = time.time()

# Extract the final processed amplitude estimation
quantum_estimate = result.estimation_processed

print("Quantum-estimated final amount: ${:,.0f}".format(quantum_estimate))
print("Quantum estimation time: {:.4f} seconds, evaluations used: {}".format(
    end_time_q - start_time_q, 2**num_eval_qubits
))

ModuleNotFoundError: No module named 'qiskit_aer'